In [102]:
import os
import pandas as pd

cc18_info = pd.read_csv("../package_metadata/cc18_information.csv", sep=";")
cc18_info['task'] = "classification"
ctr23_info = pd.read_csv("../package_metadata/ctr23_information.csv", sep=";")
ctr23_info['task'] = "regression"

cc18_ctr23_info = pd.concat([cc18_info, ctr23_info], ignore_index=True, join='outer')
cc18_ctr23_info = cc18_ctr23_info.fillna("no info")

In [103]:
def read_all_experiment_results_data():
    root_path = "../package_metadata/openml"
    target_folders = ["kernels_iid_comparison", "other_baselines"]

    experiment_results = []

    if os.path.exists(root_path):
        for dataset_name in os.listdir(root_path):
            dataset_full_path = os.path.join(root_path, dataset_name)
            if os.path.isdir(dataset_full_path):
                for folder_name in target_folders:
                    target_path = os.path.join(dataset_full_path, folder_name)
                    if os.path.exists(target_path):
                        for filename in os.listdir(target_path):
                            if filename.endswith(".csv"):
                                file_path = os.path.join(target_path, filename)
                                try:
                                    df = pd.read_csv(file_path)

                                    # EXTRACT MODEL NAME FROM FILENAME ---
                                    if "_ann" in filename:
                                        df["model_name"] = "nn"
                                    elif "_xgboost" in filename:
                                        df["model_name"] = "xgboost"
                                    else:
                                        df["model_name"] = "unknown"

                                    # --- EXISTING COLUMNS LOGIC ---
                                    if "data_modification_method" not in df.columns:
                                        df["data_modification_method"] = "none"

                                    df["dataset_name"] = dataset_name
                                    df['Group'] = df['explainer'].apply(
                                        lambda x: 'large' if x == 'expected_gradients' else 'small'
                                    )
                                    experiment_results.append(df)

                                except Exception as e:
                                    print(f"Error reading file {file_path}: {e}")

    if experiment_results:
        combined_df = pd.concat(experiment_results, ignore_index=True)
        return combined_df
    else:
        return pd.DataFrame()

In [104]:
results = read_all_experiment_results_data()

Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_700_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_600_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_400_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_500_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_1000_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradients_na_ann_900_1.csv: No columns to parse from file
Error reading file ../package_metadata/openml/nomao/other_baselines/influence_none_expected_gradien

In [105]:
results = pd.merge(
    results,
    cc18_ctr23_info,
    left_on='dataset_name',
    right_on='name',
    how='left'
)

In [106]:
from goodpoints.compress import largest_power_of_four

results['compression_coefficient'] = results.apply(
    lambda row: row['size'] / (largest_power_of_four(row['n_samples_test']) ** 0.5),
    axis=1
)

In [107]:
def get_explainer_total(row):
    ex = row['explainer']
    st = row['strategy']

    if ex == 'expected_gradients':
        return 'expected_gradients'

    elif ex == 'shap' and st == 'kernel':
        return 'shap_kernel'

    elif ex == 'shapiq' and st == 'kernel':
        return 'shapiq_kernel'

    elif ex == 'sage' and st == 'permutation':
        return 'sage_permutation'

results['explainer_total'] = results.apply(get_explainer_total, axis=1)

In [108]:
def get_method_total(row):
    method = row['method']
    kernel = row['kernel']
    data_modification_method = row['data_modification_method']

    if method == 'iid':
        return 'iid'

    if method == 'arfpy':
        return 'arfpy'

    if method == 'stein_thinning':
        return 'stein_thinning'

    if method == 'influence':
        return 'influence'

    if method == 'kernel_thinning' and data_modification_method == "none":
        return 'kt_' + kernel

    if method == 'kernel_thinning' and data_modification_method != "none":
        return 'kt_' + data_modification_method

    return None

results['method_total'] = results.apply(get_method_total, axis=1)


In [109]:
results['method_total'].unique()

array(['kt_gaussian', 'kt_sobolev', 'kt_inverse_multiquadric',
       'kt_matern', 'iid', 'arfpy', 'kt_predictions', 'kt_stratified',
       'stein_thinning', 'influence'], dtype=object)

In [110]:
aggregation_columns = ['mae', 'top_k', 'explanation_time', 'mmd', 'compression_time', 'unique_samples']
group_columns = [col for col in results.columns if col not in aggregation_columns]

grouped = results.groupby(group_columns, as_index=False).agg({
    'mae': ['mean', 'std'],
    'top_k': ['mean', 'std'],
    'explanation_time': ['mean', 'std'],
    'mmd': ['mean', 'std'],
    'compression_time': ['mean', 'std'],
    'unique_samples': ['mean', 'std']
})

grouped.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col
                   for col in grouped.columns]

grouped = grouped.rename(columns={
    col: col.replace('mean', 'mean').replace('std', 'std')
    for col in grouped.columns
})

In [111]:
grouped.to_csv("../package_metadata/results_aggregation.csv", index=False) # each experiment is 1 row - but aggreagted over repeats
results.to_csv("../package_metadata/results_all.csv", index=False) # each experiment with 1 repeat is 1 row